In [19]:
import sys, os
sys.path.append(os.path.abspath("../../"))
from src.dataset.data_loader import load_competition_data

train_dataset, test_dataset, sample_submission = load_competition_data()
print("Train shape:", train_dataset.shape)
print("Test shape :", test_dataset.shape)

Train shape: (2000, 7)
Test shape : (500, 7)


In [20]:
import string
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import sys
import io
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [21]:
# handles missing data first for tfidfVectorizer usage

train_df = train_dataset.copy()

print("=== standard NaN check =======")
print(train_df.isna().sum())

placeholders = {"n/a", "na", "null", "none", "nan", "-", "--", "?"}

def text_missing(col):
    if col.dtype != "object":          # ← skip numeric columns
        return pd.Series({
            "is_nan": col.isna().sum(),
            "empty_string": 0,
            "placeholder": 0,
        })
    s = col.str.strip()
    return pd.Series({
        "is_nan": col.isna().sum(),
        "empty_string": (s == "").sum(),
        "placeholder": s.str.lower().isin(placeholders).sum(),
    })

print("\nText-quality audit:")
print(train_df.apply(text_missing).T)

=== standard NaN check =======
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Text-quality audit:
        is_nan  empty_string  placeholder
prompt       0             0            0
A            0             0            0
B            0             0            0
C            0             0            0
D            0             0            0
E            0             0            0
answer       0             0            0


In [22]:
# basic cleaning of text -

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train_df[col] = train_df[col].str.strip().str.replace(r'\s+', ' ', regex=True)

print("Cleaning done — sample prompt:")
print(train_df['prompt'].iloc[0])

Cleaning done — sample prompt:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [23]:
_stdout = sys.stdout; sys.stdout = io.StringIO()  # temporarily silence nltk's verbose download messages
nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('stopwords')
sys.stdout = _stdout

stop_words = set(stopwords.words('english'))

def tokenize(text, remove_stopwords=False):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # remove punctuation, keep alphanumeric
    text = re.sub(r'\s+', ' ', text).strip()    # normalise whitespace
    tokens = word_tokenize(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    return tokens

# TF-IDF: tokens joined back to string (vectorizer expects string input)
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train_df[f'{col}_tfidf'] = train_df[col].apply(
        lambda x: ' '.join(tokenize(x, remove_stopwords=True))
    )

print("Tokenization complete.")
print("TF-IDF sample :", train_df['prompt_tfidf'].iloc[0])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ais\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ais\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ais\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Tokenization complete.
TF-IDF sample : pick best possible answer martin heidegger view relationship time human existence among listed options


In [24]:
# Fit on all text to build a shared vocabulary

all_text = pd.concat([train_df[f'{col}_tfidf'] for col in ['prompt','A','B','C','D','E']])
vectorizer = TfidfVectorizer()
vectorizer.fit(all_text)

print(f"TF-IDF vocabulary size: {len(vectorizer.vocabulary_):,} terms")

# Cosine similarity: prompt vs each option → pick top 3
def get_top3_tfidf(row):
    prompt_vec = vectorizer.transform([row['prompt_tfidf']])
    scores = {
        opt: cosine_similarity(prompt_vec, vectorizer.transform([row[f'{opt}_tfidf']]))[0][0]
        for opt in ['A','B','C','D','E']
    }
    return ' '.join(sorted(scores, key=scores.get, reverse=True)[:3])

train_df['tfidf_pred'] = train_df.apply(get_top3_tfidf, axis=1)
print("\nSample predictions (actual answer | tfidf_pred):")
print(train_df[['answer','tfidf_pred']].head(5).to_string())

TF-IDF vocabulary size: 2,842 terms

Sample predictions (actual answer | tfidf_pred):
  answer tfidf_pred
0      B      C D B
1      A      C B A
2      C      A B C
3      B      C D B
4      A      D E B


In [25]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    return 1.0 / (predicted.index(actual) + 1)

def mapk(df, pred_col):
    return df.apply(lambda row: apk(row['answer'], row[pred_col].split()), axis=1).mean()

map3_tfidf = mapk(train_df, 'tfidf_pred')
print(f"TF-IDF MAP@3 : {map3_tfidf:.4f}")

TF-IDF MAP@3 : 0.3128


In [26]:
OPTIONS = ['A', 'B', 'C', 'D', 'E']

# -------- Q1: Frequency distribution of correct answers ---------
freq = train_df['answer'].value_counts()
print("=== Q1: Answer Frequency Distribution ===")
print(freq.to_string())
most_frequent_count  = freq.iloc[0]
least_frequent_count = freq.iloc[-1]
q1_answer = most_frequent_count + least_frequent_count
print(f"\nMost frequent  : {freq.index[0]}  →  {most_frequent_count}")
print(f"Least frequent : {freq.index[-1]}  →  {least_frequent_count}")
print(f"Sum (Q1 answer): {q1_answer}")

# --------- Q2: Unique words in cleaned prompt column ------------------
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

cleaned_prompts = train_df['prompt'].apply(clean_text)
all_words = set()
for prompt in cleaned_prompts:
    all_words.update(prompt.split())

q2_answer = len(all_words)
print(f"\n=== Q2: Vocabulary size (unique words in cleaned prompts) ===")
print(f"Vocabulary size: {q2_answer}")

# -------- Q3: Words left after stop word removal for Row ID 1 -----------
row1_prompt = cleaned_prompts.iloc[0]
row1_words  = row1_prompt.split()
filtered    = [w for w in row1_words if w not in ENGLISH_STOP_WORDS]
q3_answer   = len(filtered)
print(f"\n=== Q3: Words in Row ID 1 prompt after stop word removal ===")
print(f"Cleaned prompt of row 1 : {row1_prompt}")
print(f"Words before removing stop words : {row1_words}")
print(f"Words after removing stop words : {filtered}")
print(f"Count (Q3 answer): {q3_answer}")

# Q4: TF-IDF vocabulary size (prompt + all options combined)
all_texts = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_texts.extend(train_df[col].tolist())

vectorizer_q4 = TfidfVectorizer(stop_words='english')
vectorizer_q4.fit(all_texts)
q4_answer = len(vectorizer_q4.vocabulary_)
print(f"\n=== Q4: TF-IDF vocabulary size (all prompts + options) ===")
print(f"Feature columns (Q4 answer): {q4_answer}")

# Q5: Cosine similarity between prompt and option A for Row ID 1
row1 = train_df.iloc[0]
vecs = vectorizer_q4.transform([row1['prompt'], row1['A']])
q5_answer = cosine_similarity(vecs[0], vecs[1])[0][0]
print(f"\n=== Q5: Cosine similarity — Row ID 1, prompt vs option A ===")
print(f"Prompt   : {row1['prompt']}")
print(f"Option A : {row1['A']}")
print(f"Cosine similarity (Q5 answer): {q5_answer:.4f}")

# Q6: % of rows where highest cosine sim option = correct answer
def tfidf_top1(row):
    texts = [row['prompt']] + [row[o] for o in OPTIONS]
    vecs = vectorizer_q4.transform(texts)
    prompt_vec = vecs[0]
    sims = {o: cosine_similarity(prompt_vec, vecs[i+1])[0][0] for i, o in enumerate(OPTIONS)}
    return max(sims, key=sims.get)

train_df['tfidf_top1'] = train_df.apply(tfidf_top1, axis=1)
q6_answer = (train_df['tfidf_top1'] == train_df['answer']).mean() * 100
print(f"\n=== Q6: % rows where top TF-IDF cosine sim = correct answer ===")
print(f"Accuracy (Q6 answer): {q6_answer:.2f}%")

# Q7: MAP@3 if ground truth = C and prediction = C A B
q7_answer = 1.0 / ((['C', 'A', 'B'].index('C')) + 1)
print(f"\n=== Q7: MAP@3 — ground truth C, prediction C A B ===")
print(f"C is at position 1 → 1/1 = {q7_answer:.4f}")

# Q8: MAP@3 if ground truth = B and prediction = D B E
q8_answer = 1.0 / ((['D', 'B', 'E'].index('B')) + 1)
print(f"\n=== Q8: MAP@3 — ground truth B, prediction D B E ===")
print(f"B is at position 2 → 1/2 = {q8_answer:.4f}")

# Q9: Majority class baseline MAP@3
top3_answers = freq.index[:3].tolist()
static_pred = ' '.join(top3_answers)
train_df['majority_pred'] = static_pred

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    return 1.0 / (predicted.index(actual) + 1)

def mapk(df, pred_col):
    return df.apply(lambda row: apk(row['answer'], row[pred_col].split()), axis=1).mean()

q9_answer = mapk(train_df, 'majority_pred')
print(f"\n=== Q9: Majority class baseline MAP@3 ===")
print(f"Static prediction for every row: '{static_pred}'")
print(f"MAP@3 (Q9 answer): {q9_answer:.4f}")

# Q10: TF-IDF pipeline MAP@3 across entire train set (reuses map3_tfidf from earlier task cell)
print(f"\n=== Q10: TF-IDF pipeline MAP@3 (entire train set) ===")
print(f"MAP@3 (Q10 answer): {map3_tfidf:.4f}")

=== Q1: Answer Frequency Distribution ===
answer
B    490
C    459
A    369
D    358
E    324

Most frequent  : B  →  490
Least frequent : E  →  324
Sum (Q1 answer): 814

=== Q2: Vocabulary size (unique words in cleaned prompts) ===
Vocabulary size: 859

=== Q3: Words in Row ID 1 prompt after stop word removal ===
Cleaned prompt of row 1 : pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options
Words before removing stop words : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
Words after removing stop words : ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Count (Q3 answer): 13

=== Q4: TF-IDF vocabulary size (all prompts + options) ===
Feature columns (Q4 answ

In [27]:
print("\n" + "="*55)
print("MILESTONE 1 Q/A — ANSWERS SUMMARY")
print("="*55)
print(f"  Q1  — most + least frequent count sum : {q1_answer}")
print(f"  Q2  — vocabulary size (prompts)       : {q2_answer}")
print(f"  Q3  — words after stopword removal    : {q3_answer}")
print(f"  Q4  — TF-IDF feature columns          : {q4_answer}")
print(f"  Q5  — cosine sim row1 prompt vs A     : {q5_answer:.4f}")
print(f"  Q6  — top-1 accuracy TF-IDF           : {q6_answer:.2f}%")
print(f"  Q7  — MAP@3 (C predicted as C A B)    : {q7_answer:.4f}")
print(f"  Q8  — MAP@3 (B predicted as D B E)    : {q8_answer:.4f}")
print(f"  Q9  — majority class baseline MAP@3   : {q9_answer:.4f}")
print(f"  Q10 — TF-IDF pipeline MAP@3           : {map3_tfidf:.4f}")
print("="*55)


MILESTONE 1 Q/A — ANSWERS SUMMARY
  Q1  — most + least frequent count sum : 814
  Q2  — vocabulary size (prompts)       : 859
  Q3  — words after stopword removal    : 13
  Q4  — TF-IDF feature columns          : 2762
  Q5  — cosine sim row1 prompt vs A     : 0.2328
  Q6  — top-1 accuracy TF-IDF           : 13.70%
  Q7  — MAP@3 (C predicted as C A B)    : 1.0000
  Q8  — MAP@3 (B predicted as D B E)    : 0.5000
  Q9  — majority class baseline MAP@3   : 0.4213
  Q10 — TF-IDF pipeline MAP@3           : 0.3128
